# Import libraries

In [ ]:
from sqlalchemy import text
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from h5n1.db import get_engine
from transformers import pipeline

# User Input for Date Range

Follow the prompt here. For testing answer N and input start and end date. For full data, answer Y.

In [ ]:
# yes or no for full retrieval
i = 0
while i == 0:
    user_input = input("Retrieve Full x_post_unique Table? Answer:(Y or N)")
    if user_input == 'N' or user_input == 'n':
        start_date = input("Type Start Date in YYYY-MM-DD format")
        end_date = input("Type END Date in YYYY-MM-DD format")
        QUERY = f"SELECT post_id, created_at, lang, post_text FROM x_post_unique WHERE created_at >= '{start_date}' AND created_at < '{end_date}';"
        i = 1
    elif user_input == 'Y' or user_input == 'y':
        QUERY = f"SELECT post_id, created_at, lang, post_text FROM x_post_unique;"
        i = 1
    else:
        print('Incorrect Input')
# read the query
df = pd.read_sql(QUERY, get_engine())
print('Retrieval Successful')
print(f'The start date in the dataset: {df['created_at'].min()}')
print(f'The end date in the dataset: {df['created_at'].max()}')

# Clean the Post Text

In [ ]:
# remove \n from the text 
new_text = []
for index, row in df.iterrows():
    new_text.append(df.iloc[index]['post_text'].replace('\n', ' '))
df['cleaned_post_text'] = new_text

# drop the post text column
df.drop(columns='post_text', inplace=True)

print('Added Column of Cleaned Post Text and Drop Post Columns Successful')

# Produce The Sentiments

In [ ]:
# pipeline sentiment with xlm roberta base with three scores
model_path = "cardiffnlp/twitter-xlm-roberta-base-sentiment"
sentiment_3_score = pipeline("sentiment-analysis", model=model_path, top_k=None)

# extract only text and fill na with quotation
texts = df['cleaned_post_text'].fillna('').astype(str).tolist()
print('Extract Text to List Successful')

# feeding text into the model
results = sentiment_3_score(texts, batch_size=64, truncation=True, max_length=512)
print('Produce Sentiment Pipeline Successful')

# Adding scores to the dataframe
df['negative_score'] = [i['score'] for r in results for i in r if i['label'] == 'negative']
df['neutral_score'] = [i['score'] for r in results for i in r if i['label'] == 'neutral']
df['positive_score'] = [i['score'] for r in results for i in r if i['label'] == 'positive']
print('Added Negative, Neutral, Positive scores Sucessful')

# Calculate the final sentiment
cols = {'negative_score': 'negative', 'neutral_score': 'neutral', 'positive_score': 'positive'}
df['final_sentiment'] = df[list(cols)].idxmax(axis=1).map(cols)
print('Calculation of Final Sentiment Successful')

# Write the Dataframe to Database

In [ ]:
# write the dataframe to a new table
df.to_sql(
    "x_post_sentiment",   # table name
    get_engine(),
    if_exists="fail",     # see below
    index=False,          # don't write the pandas row index as an extra column
    chunksize=10_000,     # send rows in batches instead of all at once
    method="multi",       # several rows per INSERT, which is much faster over the proxy
)
print('Write the Database Successful')

In [ ]:
# test if the table exist and retrievable
df_1 = pd.read_sql("SELECT * FROM x_post_sentiment LIMIT 10;", get_engine())
df_1

In [ ]:
# If need to drop the table
# with get_engine().begin() as conn:
#     conn.execute(text("DROP TABLE IF EXISTS x_post_sentiment"))